<div style="text-align: center;">

# Sustainability Aware Asset Management  
# Portfolio Allocation with a Carbon Objective  

<br>

**Course:** Sustainability Aware Asset Management<br>  
**Instructor:** Professor Eric Jondeau<br> 
**Academic Year:** 2025–2026<br>  

<br>

**Group Members**  
Matteo Piras<br>  
Tomas Papuga<br> 
Marco d'Amico<br>
Roberto Berardi<br> 

<br>

**Submission Date:** May 2026  

</div>

# Part I- Standard Portfolio Allocation
## 1) Data Cleaning

### Missing Prices and Datastream Error Rows

The raw datasets are direct exports from Datastream and may contain structural inconsistencies that must be addressed before any financial analysis.

Some ISIN codes are flagged with `$$ER`, indicating that Datastream could not correctly match the requested security (for example, due to share class mismatches or unavailable data). These rows do not contain valid financial information and are therefore removed from all datasets.

We then standardize the structure of each file by:
- Converting ISIN codes to a consistent string format,
- Removing non-numeric columns such as firm names,
- Transposing the data so that dates form the index and firms form the columns,
- Converting all values to numeric format.

At this stage, we only correct structural and data-quality issues. No economic filtering (region, carbon availability, liquidity, etc.) is applied yet. 

In [55]:
import pandas as pd
import numpy as np

print("Starting finalized data preparation process...")

# --------------------------------------------------------
# 1. Setup and Static Data Filtering
# --------------------------------------------------------
# Load the static dataset to identify the initial investment universe
static_df = pd.read_excel('Data_2026/Static_2025.xlsx')
static_df['ISIN'] = static_df['ISIN'].astype(str).str.strip()

# Isolate firms belonging to the Emerging Markets (EM) region
em_firms = static_df[static_df['Region'] == 'EM'].copy()
valid_em_isins = em_firms['ISIN'].unique().tolist()

def load_and_transpose(filepath, is_monthly=False):
    """
    Standardizes Datastream Excel exports into time-series DataFrames.
    Removes error flags ($$ER) and transposes firms to columns.
    """
    df = pd.read_excel(filepath)
    # Rule: Explicitly delete ISINs with Datastream errors ($$ER)
    df = df[~df['ISIN'].astype(str).str.contains(r'\$\$ER', na=False)]
    df['ISIN'] = df['ISIN'].astype(str).str.strip()
    df.set_index('ISIN', inplace=True)
    
    if 'NAME' in df.columns:
        df.drop(columns=['NAME'], inplace=True)
    
    df_t = df.T
    df_t = df_t.apply(pd.to_numeric, errors='coerce')
    
    # Handle indexing based on data frequency (Monthly for RI, Yearly for Carbon)
    if is_monthly:
        df_t.index = pd.to_datetime(df_t.index)
    else:
        df_t.index = df_t.index.astype(int)
    return df_t

print("Static data loaded and EM firms identified.")

Starting finalized data preparation process...
Static data loaded and EM firms identified.


### Missing Values: Beginning, Middle, and End of Sample

The datasets contain different types of missing observations that must be handled carefully to ensure consistent return estimation and avoid distortions in portfolio construction.

First, missing values at the beginning of the sample typically correspond to firms that were not yet listed or had not started reporting data. In this case, no correction is applied.

Second, missing values between two available observations usually reflect temporary reporting gaps or data issues. To preserve continuity in the return series, we apply a forward-fill procedure to bridge these internal gaps.

Third, missing values at the end of the sample generally correspond to firm delisting or default events. In such cases, the price is assumed to drop to zero, implying a realized return of −100% in the month following the last valid observation. All subsequent periods are set to missing values to avoid artificially lowering volatility.

This treatment ensures a realistic handling of firm exits while maintaining the integrity of the historical return series.

In [56]:
# --------------------------------------------------------
# 2. Return Index (RI) Processing with Delisting Logic
# --------------------------------------------------------
ri_monthly = load_and_transpose('Data_2026/DS_RI_T_USD_M_2025.xlsx', is_monthly=True)
ri_em = ri_monthly[[isin for isin in valid_em_isins if isin in ri_monthly.columns]].copy()

# Rule: Treat prices < 0.5 as missing values to avoid extreme returns from penny stocks
ri_em[ri_em < 0.5] = np.nan

# CRITICAL: Identify the real last valid date BEFORE filling gaps.
# This ensures that delisting (final disappearance) is not masked by forward-filling.
real_last_valid_dates = ri_em.apply(lambda col: col.last_valid_index())

# Rule: Fill gaps between available values (Forward Fill) to bridge misreporting
ri_em_filled = ri_em.ffill()
returns_em = ri_em_filled.pct_change()

# Applying Precise Delisting Logic: -100% loss followed by NaNs
for isin in returns_em.columns:
    last_date = real_last_valid_dates[isin]
    if pd.notna(last_date):
        last_pos = returns_em.index.get_loc(last_date)
        
        # If the firm disappears before the end of the sample, acknowledge the loss
        if last_pos < len(returns_em) - 1:
            # Force -100% return in the month following the last valid price
            returns_em.iloc[last_pos + 1, returns_em.columns.get_loc(isin)] = -1.0
            
            # Ensure post-delisting periods are NaN (not zero) to avoid stale price/low volatility bias
            if last_pos + 1 < len(returns_em) - 1:
                returns_em.iloc[last_pos + 2:, returns_em.columns.get_loc(isin)] = np.nan
    else:
        # Mark firms with no valid data for complete removal
        returns_em[isin] = np.nan

### Low Prices, Stale Prices, and Carbon Availability

Additional filters are applied to ensure that return calculations, risk estimates, and sustainability constraints are not distorted by data irregularities.

First, very low prices can generate extreme or unstable returns. When the total return index (RI) falls below 0.5, percentage returns may become excessively large or undefined. For this reason, all price observations below 0.5 are treated as missing values. If the price is missing at the end of year 
𝑌
Y, the firm is excluded from the investment set for year 
𝑌
+
1
Y+1.

Second, stale prices may artificially reduce estimated volatility. If a firm’s price does not change for a prolonged period, returns are equal to zero, which may bias risk estimates. To address this issue, we compute the proportion of zero monthly returns over the 10-year estimation window. Firms with more than 50% zero returns are considered illiquid and excluded from the subsequent holding period.

Finally, to ensure consistency between the financial and sustainability analysis, firms must have available Scope 1 and Scope 2 carbon data at the end of year 
𝑌
Y to be eligible for inclusion in the investment set.

These filters ensure a reliable and economically meaningful investment universe.

In [57]:
# Rule: Treat prices < 0.5 as missing values to avoid extreme returns from penny stocks
ri_em[ri_em < 0.5] = np.nan


scope1_em = load_and_transpose('Data_2026/DS_CO2_SCOPE_1_Y_2025.xlsx', is_monthly=False)
scope2_em = load_and_transpose('Data_2026/DS_CO2_SCOPE_2_Y_2025.xlsx', is_monthly=False)

# Filter for EM and apply Forward Fill for reporting gaps between years
scope1_em = scope1_em[[isin for isin in valid_em_isins if isin in scope1_em.columns]].copy().ffill()
scope2_em = scope2_em[[isin for isin in valid_em_isins if isin in scope2_em.columns]].copy().ffill()

# Rule: Physically delete firms with no associated data from all tables
returns_em.dropna(how='all', axis=1, inplace=True)
scope1_em.dropna(how='all', axis=1, inplace=True)
scope2_em.dropna(how='all', axis=1, inplace=True)

print("Data Validation Overview:")
print(f"- Returns Matrix: {returns_em.shape[0]} months, {returns_em.shape[1]} firms")
print(f"- Scope 1/2 Matrices: {scope1_em.shape[0]} years")
display(returns_em.head())

Data Validation Overview:
- Returns Matrix: 314 months, 668 firms
- Scope 1/2 Matrices: 27 years


ISIN,ARALUA010258,ARP125991090,ARSIDE010029,BMG211591018,BRABEVACNOR1,BRBBASACNOR3,BRBBDCACNPR8,BRBRFSACNOR8,BRBRKMACNPA4,BRCESPACNPB4,...,ZAE000117321,ZAE000127148,ZAE000134961,ZAE000170049,ZAE000179420,ZAE000191342,ZAE000255915,ZAE000298253,ZAE000302618,ZAE000322095
1999-12-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2000-01-31,0.176511,-0.051861,-0.004877,NaN,-0.044175,-0.067245,0.023986,-0.249634,0.011906,NaN,...,-0.115161,-0.049527,0.096965,0.067794,0.205596,0.099568,-0.080722,0.125341,0.021174,-0.114179
2000-02-29,0.050002,0.178923,0.039543,NaN,-0.045539,-0.036328,-0.107794,0.005452,0.043492,NaN,...,-0.037800,-0.100608,-0.326248,0.028928,-0.053481,-0.084315,-0.062844,-0.181598,0.018379,-0.023150
2000-03-31,-0.023552,-0.105072,-0.002113,NaN,0.313941,-0.130625,0.152061,0.014002,0.002361,NaN,...,-0.067887,-0.034040,-0.044066,-0.176088,-0.126866,0.073915,-0.054382,-0.038462,-0.142989,-0.104274
2000-04-28,-0.016173,-0.161842,-0.071188,NaN,-0.001350,-0.077311,-0.108617,-0.032966,-0.020143,NaN,...,-0.034586,-0.042240,-0.172342,-0.129131,-0.035409,-0.208833,-0.030503,-0.224615,-0.095572,0.003496


### Dynamic Investment Set Construction (10-Year Rolling Window)

After cleaning the data, we construct the investment universe dynamically for each portfolio formation year.

For each year 
𝑌
Y, we use a 10-year rolling estimation window (from January 
𝑌
−
9
Y−9 to December 
𝑌
Y) to determine firm eligibility. The portfolio formed at the end of year 
𝑌
Y is implemented during year 
𝑌
+
1
Y+1.

A firm is included in the investment set only if it satisfies all of the following conditions:

Price availability at formation date:
The firm must have a valid price at the end of year 
𝑌
Y. Firms with missing or invalid prices at the portfolio formation date are excluded.

Liquidity condition (stale price filter):
Over the 10-year estimation window, the proportion of zero monthly returns must not exceed 50%. This ensures that illiquid firms do not artificially reduce estimated portfolio volatility.

Carbon data availability:
Both Scope 1 and Scope 2 emissions data must be available at the end of year 
𝑌
Y. This guarantees consistency between the financial allocation and the sustainability analysis.

The resulting set of eligible firms is stored for each holding year 
𝑌
+
1
Y+1, forming a sequence of dynamically updated investment universes.

In [58]:
print("Initiating dynamic filtering for 10-year rolling windows...")

# Step 5: Dynamic Universe Construction
# We identify eligible firms for each year based on historical liquidity and ESG disclosure.
valid_investment_sets = {}
estimation_years = range(2009, 2025)

for Y in estimation_years:
    # Define the 10-year rolling estimation window (Dec Y-9 to Dec Y)
    start_date, end_date = f"{Y-9}-01-01", f"{Y}-12-31"
    window_returns = returns_em.loc[start_date:end_date]
    window_prices = ri_em.loc[start_date:end_date]
    
    valid_isins_for_year = []
    
    for isin in returns_em.columns:
        # Rule: Price Availability at end of year Y
        # Exclude firms missing a valid price (>0.5 USD) at the portfolio formation date.
        if pd.isna(window_prices[isin].iloc[-1]):
            continue
            
        # Rule: Stale Price / Liquidity Filter
        # Calculate the ratio of months with zero returns to avoid illiquid assets.
        total_valid_months = window_returns[isin].notna().sum()
        if total_valid_months == 0: continue
        
        # Exclude if the firm has no price movement for > 50% of the window.
        if (window_returns[isin] == 0.0).sum() / total_valid_months > 0.5:
            continue
            
        # Rule: Carbon Data Availability for Year Y
        # Both Scope 1 and Scope 2 data must be present for the firm to be eligible.
        if Y in scope1_em.index and Y in scope2_em.index:
            if isin in scope1_em.columns and isin in scope2_em.columns:
                if not pd.isna(scope1_em.loc[Y, isin]) and not pd.isna(scope2_em.loc[Y, isin]):
                    valid_isins_for_year.append(isin)
    
    # Store the investment set for the subsequent holding period (Y + 1)
    valid_investment_sets[Y + 1] = valid_isins_for_year
    print(f"Holding Year {Y+1}: {len(valid_isins_for_year)} firms identified.")

print("\nInvestment universes constructed successfully.")

Initiating dynamic filtering for 10-year rolling windows...
Holding Year 2010: 67 firms identified.
Holding Year 2011: 160 firms identified.
Holding Year 2012: 199 firms identified.
Holding Year 2013: 225 firms identified.
Holding Year 2014: 246 firms identified.
Holding Year 2015: 268 firms identified.
Holding Year 2016: 298 firms identified.
Holding Year 2017: 335 firms identified.
Holding Year 2018: 382 firms identified.
Holding Year 2019: 418 firms identified.
Holding Year 2020: 465 firms identified.
Holding Year 2021: 505 firms identified.
Holding Year 2022: 542 firms identified.
Holding Year 2023: 572 firms identified.
Holding Year 2024: 580 firms identified.
Holding Year 2025: 567 firms identified.

Investment universes constructed successfully.
